Dataset Creation

In [3]:
import pandas as pd
import json
import random
from pathlib import Path

def generate_numeric_options(correct_value, all_values):
    """Generate 3 unique distractor values close to the correct one."""
    options = {correct_value}
    max_attempts = 100
    attempts = 0

    while len(options) < 4 and attempts < max_attempts:
        # Offset scales with ~10% of value, minimum ±3
        scale = max(3, int(correct_value * 0.1))
        candidate = correct_value + random.randint(-scale, scale)
        attempts += 1
        if candidate <= 0 or candidate == correct_value:
            continue
        options.add(candidate)

    # Fallback: small perturbations if needed
    while len(options) < 4:
        options.add(correct_value + random.choice([-1, 1]) * random.randint(1, 5))

    options = list(options)
    random.shuffle(options)
    return options

def generate_type_options(correct_type, all_types):
    """Generate 3 distractor types."""
    other_types = [t for t in all_types if t != correct_type]
    options = random.sample(other_types, 3)
    options.append(correct_type)
    random.shuffle(options)
    return options

def label_options(options):
    """Attach A), B), C), D) prefixes."""
    labels = ["A)", "B)", "C)", "D)"]
    return [f"{label} {opt}" for label, opt in zip(labels, options)]

def create_mcqa_from_excel(input_excel, output_json):
    df = pd.read_excel(input_excel)
    df.columns = [c.strip().lower() for c in df.columns]  # normalize column names

    all_types = df['type1'].unique().tolist()
    all_hp = df['hp'].tolist()
    all_defense = df['defense'].tolist()
    all_speed = df['speed'].tolist()

    mcqa_data = []

    for _, row in df.iterrows():
        name = row['name']

        # --- Type 1 ---
        type_opts = generate_type_options(row['type1'], all_types)
        type_q = {
            "question": f"What is the Type 1 of {name}?",
            "options": label_options(type_opts),
            "answer": f"{['A)','B)','C)','D)'][type_opts.index(row['type1'])]} {row['type1']}",
            "answer_full_writing": f"{row['type1']}",
            "trait": "type1"
        }
        mcqa_data.append(type_q)

        # --- HP ---
        hp_opts = generate_numeric_options(row['hp'], all_hp)
        hp_q = {
            "question": f"What is the Base HP (Hit Points) of {name}?",
            "options": label_options(hp_opts),
            "answer": f"{['A)','B)','C)','D)'][hp_opts.index(row['hp'])]} {row['hp']}",
            "answer_full_writing": f"{row['hp']}",
            "trait": "hp"
        }
        mcqa_data.append(hp_q)

        # --- Defense ---
        def_opts = generate_numeric_options(row['defense'], all_defense)
        def_q = {
            "question": f"What is the Base Defense stat of {name}?",
            "options": label_options(def_opts),
            "answer": f"{['A)','B)','C)','D)'][def_opts.index(row['defense'])]} {row['defense']}",
            "answer_full_writing": f"{row['defense']}",
            "trait": "defense"
        }
        mcqa_data.append(def_q)

        # --- Speed ---
        spd_opts = generate_numeric_options(row['speed'], all_speed)
        spd_q = {
            "question": f"What is the Base Speed stat of {name}?",
            "options": label_options(spd_opts),
            "answer": f"{['A)','B)','C)','D)'][spd_opts.index(row['speed'])]} {row['speed']}",
            "answer_full_writing": f"{row['speed']}",
            "trait": "speed"
        }
        mcqa_data.append(spd_q)

    # Output JSON
    Path(output_json).parent.mkdir(parents=True, exist_ok=True)
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(mcqa_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Generated {len(mcqa_data)} MCQA items saved to {output_json}")



In [4]:
# Example usage:
create_mcqa_from_excel("/Users/avduarte/Desktop/Doutoramento/Cadeiras/CMU/Trustworthy AI/HW2/pokemon-cleaned.xlsx", "pokemon_mcqa.json")

✅ Generated 3204 MCQA items saved to pokemon_mcqa.json
